# Major Project: Seasonal Agriculture Performance Analysis

**VOIS AICTE Batch 1 2026–2027**  
**Major Project – Seasonal Agriculture Performance Analysis**

### Project focus
This notebook performs a focused data-analytics investigation of how agricultural performance changes across seasons. It covers data understanding, cleaning and preparation, seasonal comparisons, environmental/resource relationships, economic outcomes, regional/crop consistency, unusual patterns, statistical analysis, visualization, conclusions, and data-driven recommendations.

> **Important:** The analytical questions below were developed after inspecting the supplied dataset, in line with the project brief. The results are generated from the dataset itself; no values are manually entered into the analysis.

## 1. Problem Statement

Agricultural activities are influenced by seasonal variations in environmental conditions, farming practices, resource availability and market conditions. Therefore, agricultural performance may differ from one season to another.

The purpose of this project is to analyze the supplied agricultural dataset and investigate seasonal differences by identifying meaningful patterns, trends, relationships and variations in the available data.

## 2. Project Objectives

This notebook addresses the project objectives by:

- exploring and understanding the dataset;
- cleaning and preparing the data;
- examining how agricultural performance varies across seasons;
- identifying seasonal patterns and trends;
- investigating relationships between seasonal conditions and agricultural outcomes;
- comparing relevant groups within seasons;
- identifying significant differences and unusual patterns;
- applying suitable statistical and visualization techniques;
- interpreting findings using evidence from the dataset; and
- developing conclusions and data-driven recommendations.

## 3. Key Questions Addressed

The project brief asks students to develop their own specific analytical questions after exploring the dataset. Based on the columns available in this dataset, the notebook investigates:

1. How is agricultural performance distributed across the seasons?
2. Which season performs best on yield, production, revenue and profit?
3. Which environmental characteristics change across seasons?
4. How do resource usage and water efficiency differ by season?
5. How do crop performances differ across seasons?
6. Are environmental conditions associated with yield?
7. Which farming/resource variables are associated with yield and profit?
8. How do economic outcomes vary across seasons and crops?
9. Are seasonal patterns consistent across states?
10. What unusual observations or outliers are present, and what seasonal patterns do they reveal?
11. What evidence-based conclusions and recommendations can support better seasonal agricultural planning?

## 4. Technology Used

- Python
- Pandas and NumPy for data handling and analysis
- Matplotlib and Seaborn for visualization
- SciPy for statistical testing
- Jupyter Notebook / Google Colab as the analysis environment

In [ ]:
# Import required libraries
import io
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from IPython.display import display, Markdown

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
sns.set_theme(style='whitegrid')
print('Libraries imported successfully.')

## 5. Load the Dataset

### Google Colab upload
Run the next cell and select the supplied CSV file from your computer. The code automatically detects the uploaded CSV, so the exact local filename does not have to match the notebook filename.

In [ ]:
# Upload the CSV file in Google Colab
from google.colab import files
uploaded = files.upload()

csv_files = [name for name in uploaded.keys() if name.lower().endswith('.csv')]
if not csv_files:
    raise FileNotFoundError('No CSV file was uploaded. Please upload the seasonal agriculture performance CSV.')

DATA_FILE = csv_files[0]
df = pd.read_csv(DATA_FILE)
print(f'Dataset loaded from: {DATA_FILE}')
print(f'Rows: {df.shape[0]:,} | Columns: {df.shape[1]}')

## 6. Initial Dataset Understanding

In [ ]:
print('Dataset shape:', df.shape)
print('\nColumn names:')
for i, col in enumerate(df.columns, 1):
    print(f'{i:2}. {col}')

print('\nFirst 5 rows:')
display(df.head())

print('\nData types:')
display(df.dtypes.to_frame('Data Type'))

In [ ]:
# Numerical summary
print('Numerical descriptive statistics:')
display(df.describe().T)

print('Categorical columns and number of unique values:')
cat_info = pd.DataFrame({
    'Unique_Values': df.select_dtypes(include='object').nunique(),
    'Missing_Values': df.select_dtypes(include='object').isna().sum()
})
display(cat_info)

In [ ]:
# Important category distributions
for col in ['Season', 'State', 'District', 'Crop', 'Irrigation_Method']:
    if col in df.columns:
        print(f'\n{col} distribution:')
        display(df[col].value_counts(dropna=False).to_frame('Count'))

## 7. Data Quality Assessment and Cleaning

The project brief requires the data to be cleaned and prepared before analysis. The following checks identify missing values, duplicates, data-type issues, impossible values and extreme observations. Missing numerical observations are **not automatically replaced with arbitrary values**; the analysis uses appropriate valid observations for each calculation.

In [ ]:
# Missing values and duplicate records
quality = pd.DataFrame({
    'Missing_Count': df.isna().sum(),
    'Missing_Percent': (df.isna().mean() * 100).round(2),
    'Data_Type': df.dtypes.astype(str),
    'Unique_Count': df.nunique(dropna=True)
}).sort_values('Missing_Count', ascending=False)

display(quality)
print('Duplicate rows:', df.duplicated().sum())
print('Duplicate Farm_ID values:', df['Farm_ID'].duplicated().sum())

In [ ]:
# Standardize text columns without changing their meaning
text_cols = df.select_dtypes(include='object').columns.tolist()
for col in text_cols:
    df[col] = df[col].astype('string').str.strip()

# Check for empty strings created by formatting
empty_counts = {c: int((df[c] == '').sum()) for c in text_cols}
print('Empty-string counts:', empty_counts)

# Domain/range checks based on the meaning of the supplied columns
range_rules = {
    'Farm_Area_Hectares': (0, np.inf),
    'Rainfall_mm': (0, np.inf),
    'Humidity_pct': (0, 100),
    'Soil_Moisture_pct': (0, 100),
    'Seed_Quality_Score': (0, 1),
    'Yield_Tonnes_Ha': (0, np.inf),
    'Production_Tonnes': (0, np.inf),
    'Market_Price_INR_Tonne': (0, np.inf),
    'Total_Cost_INR': (0, np.inf),
    'Revenue_INR': (0, np.inf),
    'Water_Used_m3': (0, np.inf),
    'Water_Efficiency_t_per_1000m3': (0, np.inf),
    'Disease_Pest_Risk_pct': (0, 100),
    'Soil_pH': (0, 14)
}
range_report = []
for col, (lo, hi) in range_rules.items():
    if col in df.columns:
        invalid = ((df[col] < lo) | (df[col] > hi)).sum()
        range_report.append([col, int(invalid)])
range_report = pd.DataFrame(range_report, columns=['Column', 'Out_of_Range_Count'])
display(range_report)

# Work on a cleaned copy. There are no duplicate records in the supplied dataset.
clean_df = df.drop_duplicates().copy()
print('Rows after duplicate removal:', len(clean_df))
print('No arbitrary row deletion was performed for missing numerical values; analyses use valid observations.')

## 8. Derived Measures for Interpretation

Two useful measures are derived for interpretation:

- **Profit Margin (%)** = Profit / Revenue × 100, when revenue is non-zero.
- **Production per Hectare (t/ha)** = Production / Farm Area.

The supplied dataset already contains yield and water-efficiency measures, so these are retained rather than recomputed unnecessarily.

In [ ]:
df = clean_df.copy()
df['Profit_Margin_pct'] = np.where(df['Revenue_INR'] != 0, df['Profit_INR'] / df['Revenue_INR'] * 100, np.nan)
df['Production_per_Hectare'] = np.where(df['Farm_Area_Hectares'] != 0, df['Production_Tonnes'] / df['Farm_Area_Hectares'], np.nan)

print('Derived columns added: Profit_Margin_pct, Production_per_Hectare')
display(df[['Profit_INR','Revenue_INR','Profit_Margin_pct','Production_Tonnes','Farm_Area_Hectares','Production_per_Hectare']].head())

# Question 1: How is agricultural performance distributed across the seasons?

We first compare the number of observations and core agricultural performance indicators by season.

In [ ]:
season_summary = df.groupby('Season').agg(
    Farms=('Farm_ID','count'),
    Avg_Yield_Tonnes_Ha=('Yield_Tonnes_Ha','mean'),
    Avg_Production_Tonnes=('Production_Tonnes','mean'),
    Avg_Revenue_INR=('Revenue_INR','mean'),
    Avg_Profit_INR=('Profit_INR','mean')
).sort_values('Avg_Profit_INR', ascending=False)

display(season_summary.round(2))

fig, ax = plt.subplots(figsize=(8,5))
df['Season'].value_counts().reindex(['Kharif','Rabi','Zaid']).plot(kind='bar', ax=ax)
ax.set_title('Number of Farms by Season')
ax.set_xlabel('Season')
ax.set_ylabel('Number of Farms')
plt.xticks(rotation=0)
plt.show()

### Interpretation
The table and chart establish the seasonal composition of the dataset before comparing performance. Because the number of records differs by season, later comparisons focus on averages/medians rather than raw totals alone.

# Question 2: Which season performs best on yield, production, revenue and profit?

In [ ]:
performance_cols = ['Yield_Tonnes_Ha','Production_Tonnes','Revenue_INR','Profit_INR']
season_perf = df.groupby('Season')[performance_cols].agg(['mean','median'])
display(season_perf.round(2))

fig, axes = plt.subplots(1, 2, figsize=(13,5))
sns.boxplot(data=df, x='Season', y='Yield_Tonnes_Ha', ax=axes[0])
axes[0].set_title('Yield Distribution by Season')
axes[0].set_xlabel('Season'); axes[0].set_ylabel('Yield (Tonnes/Ha)')

sns.boxplot(data=df, x='Season', y='Profit_INR', ax=axes[1])
axes[1].set_title('Profit Distribution by Season')
axes[1].set_xlabel('Season'); axes[1].set_ylabel('Profit (INR)')
plt.tight_layout(); plt.show()

# Best season according to each average indicator
best_by_metric = df.groupby('Season')[performance_cols].mean().idxmax()
display(best_by_metric.to_frame('Season with Highest Mean'))

### Interpretation
Use the mean and median together because the dataset contains highly variable economic and production values. The season with the highest mean is not automatically the season with the highest typical observation when extreme values are present.

# Question 3: Which environmental characteristics change between seasons?

In [ ]:
environment_cols = [
    'Rainfall_mm','Avg_Temperature_C','Humidity_pct','Sunlight_Hours_Day',
    'Soil_Moisture_pct','Soil_pH'
]
season_environment = df.groupby('Season')[environment_cols].mean()
display(season_environment.round(2))

fig, ax = plt.subplots(figsize=(10,6))
sns.heatmap(season_environment.T, annot=True, fmt='.2f', cmap='YlGnBu', ax=ax)
ax.set_title('Average Environmental Conditions by Season')
ax.set_xlabel('Season'); ax.set_ylabel('Environmental Variable')
plt.tight_layout(); plt.show()

### Interpretation
This comparison shows which environmental conditions differ most between seasons. These differences provide the context for the later relationship analysis with agricultural outcomes.

# Question 4: How do resource usage and water efficiency differ across seasons?

In [ ]:
resource_cols = [
    'Nitrogen_kg_ha','Phosphorus_kg_ha','Potassium_kg_ha',
    'Fertilizer_kg_ha','Pesticide_Litre_ha','Water_Used_m3',
    'Water_Efficiency_t_per_1000m3'
]
resource_summary = df.groupby('Season')[resource_cols].mean()
display(resource_summary.round(2))

fig, ax = plt.subplots(figsize=(9,5))
sns.boxplot(data=df, x='Season', y='Water_Efficiency_t_per_1000m3', ax=ax)
ax.set_title('Water Efficiency by Season')
ax.set_xlabel('Season'); ax.set_ylabel('Tonnes per 1,000 m³')
plt.show()

fig, ax = plt.subplots(figsize=(9,5))
water_by_season = df.groupby('Season')['Water_Used_m3'].mean().reindex(['Kharif','Rabi','Zaid'])
water_by_season.plot(kind='bar', ax=ax)
ax.set_title('Average Water Used by Season')
ax.set_xlabel('Season'); ax.set_ylabel('Water Used (m³)')
plt.xticks(rotation=0)
plt.show()

### Interpretation
Seasonal resource differences are evaluated together with water efficiency. A season using more water is not necessarily more efficient; efficiency must be considered separately from total water use.

# Question 5: How does crop performance differ across seasons?

We compare the main crops across seasons using average yield. This addresses the requirement to compare relevant groups within different seasons.

In [ ]:
crop_season_yield = df.pivot_table(index='Crop', columns='Season', values='Yield_Tonnes_Ha', aggfunc='mean')
display(crop_season_yield.round(2))

fig, ax = plt.subplots(figsize=(11,6))
sns.heatmap(crop_season_yield, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax)
ax.set_title('Average Yield by Crop and Season')
ax.set_xlabel('Season'); ax.set_ylabel('Crop')
plt.tight_layout(); plt.show()

# Crop-season combinations with at least 20 valid yield observations
combo_counts = df.groupby(['Crop','Season'])['Yield_Tonnes_Ha'].count()
valid_combos = combo_counts[combo_counts >= 20].index
ranked = (df.set_index(['Crop','Season']).loc[df.set_index(['Crop','Season']).index.isin(valid_combos)]
          .groupby(level=[0,1])['Yield_Tonnes_Ha'].mean().sort_values(ascending=False).head(10))
display(ranked.to_frame('Mean_Yield_Tonnes_Ha').round(2))

### Interpretation
The heatmap reveals whether a crop's performance changes across seasons and whether some crop-season combinations stand out. Minimum-count filtering is used only for the ranked comparison so that very small groups do not dominate the result.

# Question 6: Are seasonal environmental conditions associated with agricultural performance?

We examine correlations between environmental variables and yield. Correlation indicates association, not causation.

In [ ]:
env_yield = df[environment_cols + ['Yield_Tonnes_Ha']].corr(numeric_only=True)['Yield_Tonnes_Ha'].sort_values(ascending=False)
display(env_yield.to_frame('Correlation_with_Yield').round(3))

fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(df[environment_cols + ['Yield_Tonnes_Ha']].corr(numeric_only=True), annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Environmental Variables and Yield Correlation Matrix')
plt.tight_layout(); plt.show()

### Interpretation
Positive correlations indicate that higher values of a variable tend to occur with higher yield, while negative correlations indicate the opposite tendency. The strength and direction should be interpreted alongside scatter plots and seasonal differences; correlation alone does not establish causality.

# Question 7: Which farming/resource variables are associated with yield and profit?

In [ ]:
analysis_vars = [
    'Rainfall_mm','Avg_Temperature_C','Humidity_pct','Sunlight_Hours_Day',
    'Soil_pH','Soil_Moisture_pct','Nitrogen_kg_ha','Phosphorus_kg_ha',
    'Potassium_kg_ha','Fertilizer_kg_ha','Pesticide_Litre_ha',
    'Seed_Quality_Score','Water_Used_m3','Disease_Pest_Risk_pct',
    'Yield_Tonnes_Ha','Profit_INR'
]
corr = df[analysis_vars].corr(numeric_only=True)

yield_corr = corr['Yield_Tonnes_Ha'].drop('Yield_Tonnes_Ha').sort_values(key=abs, ascending=False)
profit_corr = corr['Profit_INR'].drop('Profit_INR').sort_values(key=abs, ascending=False)

print('Variables most strongly associated with Yield:')
display(yield_corr.head(10).to_frame('Correlation'))
print('Variables most strongly associated with Profit:')
display(profit_corr.head(10).to_frame('Correlation'))

fig, ax = plt.subplots(figsize=(9,5))
sns.scatterplot(data=df, x='Seed_Quality_Score', y='Yield_Tonnes_Ha', hue='Season', alpha=0.55, ax=ax)
ax.set_title('Seed Quality Score vs Yield')
plt.show()

### Interpretation
The correlation rankings identify candidate factors that move with yield or profit in this dataset. They are exploratory associations and should not be described as causal effects without a controlled study.

# Question 8: How do economic outcomes vary across seasons and crops?

In [ ]:
econ = df.groupby('Season').agg(
    Mean_Cost_INR=('Total_Cost_INR','mean'),
    Mean_Revenue_INR=('Revenue_INR','mean'),
    Mean_Profit_INR=('Profit_INR','mean'),
    Median_Profit_INR=('Profit_INR','median'),
    Mean_Profit_Margin_pct=('Profit_Margin_pct','mean')
).round(2)
display(econ)

crop_econ = df.groupby('Crop').agg(
    Mean_Revenue_INR=('Revenue_INR','mean'),
    Mean_Profit_INR=('Profit_INR','mean'),
    Median_Profit_INR=('Profit_INR','median'),
    Mean_Profit_Margin_pct=('Profit_Margin_pct','mean')
).sort_values('Mean_Profit_INR', ascending=False)
display(crop_econ.round(2))

fig, ax = plt.subplots(figsize=(10,5))
median_profit = df.groupby('Season')['Profit_INR'].median().reindex(['Kharif','Rabi','Zaid'])
median_profit.plot(kind='bar', ax=ax)
ax.set_title('Median Profit by Season')
ax.set_xlabel('Season'); ax.set_ylabel('Median Profit (INR)')
plt.xticks(rotation=0)
plt.show()

### Interpretation
Economic performance is evaluated using both mean and median profit. This is important because the supplied data contain negative profits and large positive values, making the median useful for describing a more typical observation.

# Question 9: Are seasonal patterns consistent across states?

The project brief specifically asks whether seasonal patterns remain consistent across regions/categories. Here, states are used as the regional grouping.

In [ ]:
state_season = df.pivot_table(index='State', columns='Season', values='Yield_Tonnes_Ha', aggfunc='mean')
display(state_season.round(2))

# Count states where each season has the highest average yield, among available seasons
winner_counts = state_season.idxmax(axis=1).value_counts().to_frame('States_where_season_has_highest_mean_yield')
display(winner_counts)

fig, ax = plt.subplots(figsize=(12,6))
state_season.plot(kind='bar', ax=ax)
ax.set_title('Average Yield by State and Season')
ax.set_xlabel('State'); ax.set_ylabel('Mean Yield (Tonnes/Ha)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout(); plt.show()

### Interpretation
If the same season ranks highly across many states, the seasonal pattern is more consistent geographically. If rankings vary substantially by state, local conditions, crop mix or farming practices may be contributing to the differences.

# Question 10: What significant seasonal differences can be supported statistically?

For three seasons, a one-way ANOVA tests whether group means differ statistically. Because agricultural variables can be skewed and contain extreme observations, a Kruskal–Wallis test is also reported as a non-parametric check.

**Null hypothesis (H₀):** the distributions/means are the same across seasons.  
**Alternative hypothesis (H₁):** at least one season differs.

In [ ]:
def seasonal_tests(column):
    groups = [g[column].dropna().values for _, g in df.groupby('Season')]
    anova = stats.f_oneway(*groups)
    kruskal = stats.kruskal(*groups)
    return pd.Series({
        'ANOVA_F': anova.statistic,
        'ANOVA_p': anova.pvalue,
        'Kruskal_H': kruskal.statistic,
        'Kruskal_p': kruskal.pvalue
    })

test_cols = ['Yield_Tonnes_Ha','Profit_INR','Revenue_INR','Water_Used_m3','Water_Efficiency_t_per_1000m3','Disease_Pest_Risk_pct']
test_results = pd.DataFrame({c: seasonal_tests(c) for c in test_cols}).T
test_results['ANOVA_Significant_5pct'] = test_results['ANOVA_p'] < 0.05
test_results['Kruskal_Significant_5pct'] = test_results['Kruskal_p'] < 0.05
display(test_results.round(6))

print('Interpretation rule: p < 0.05 provides evidence against H0 at the 5% significance level.')

### Interpretation
A statistically significant result indicates that the observed seasonal groups are unlikely to have identical distributions/means under the test assumptions. Statistical significance does not by itself indicate practical importance or causality.

# Question 11: What unusual or unexpected seasonal patterns are present?

Outliers are investigated using the IQR rule for key numeric performance variables. Extreme values are **reported rather than silently deleted**, because the project brief explicitly asks for unusual patterns.

In [ ]:
def iqr_outliers(data, column):
    s = data[column].dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5*iqr, q3 + 1.5*iqr
    mask = (data[column] < lower) | (data[column] > upper)
    return mask, lower, upper

outlier_report=[]
for col in ['Yield_Tonnes_Ha','Production_Tonnes','Profit_INR','Water_Efficiency_t_per_1000m3']:
    mask, lower, upper = iqr_outliers(df, col)
    outlier_report.append([col, int(mask.sum()), lower, upper])
    print(f'\n{col}: {mask.sum()} IQR outliers | lower={lower:.2f}, upper={upper:.2f}')
    display(df.loc[mask, ['Farm_ID','State','District','Crop','Season',col]].sort_values(col, ascending=False).head(10))

outlier_report = pd.DataFrame(outlier_report, columns=['Variable','Outlier_Count','IQR_Lower_Bound','IQR_Upper_Bound'])
display(outlier_report.round(2))

### Interpretation
An outlier is an observation that is unusually distant from the middle 50% of the data under the IQR rule. It is not automatically an error. Such observations should be investigated in context because they can represent genuinely unusual farms, high production, unusual profitability, or data-quality issues.

# Question 12: Which irrigation methods are associated with different seasonal outcomes?

In [ ]:
irrigation_summary = df.groupby(['Season','Irrigation_Method']).agg(
    Farms=('Farm_ID','count'),
    Mean_Yield=('Yield_Tonnes_Ha','mean'),
    Mean_Water_Used=('Water_Used_m3','mean'),
    Mean_Water_Efficiency=('Water_Efficiency_t_per_1000m3','mean'),
    Mean_Profit=('Profit_INR','mean')
).reset_index()
display(irrigation_summary.round(2))

fig, ax = plt.subplots(figsize=(11,6))
sns.barplot(data=irrigation_summary, x='Irrigation_Method', y='Mean_Water_Efficiency', hue='Season', ax=ax)
ax.set_title('Mean Water Efficiency by Irrigation Method and Season')
ax.set_xlabel('Irrigation Method'); ax.set_ylabel('Tonnes per 1,000 m³')
plt.xticks(rotation=20)
plt.tight_layout(); plt.show()

### Interpretation
This comparison adds a farming-practice dimension to the seasonal analysis. The result should be interpreted as an association because the dataset is observational; irrigation method is not randomly assigned.

# 13. Consolidated Seasonal Comparison

This section brings the main seasonal indicators together so the overall pattern can be reviewed in one place.

In [ ]:
key_indicators = [
    'Yield_Tonnes_Ha','Production_Tonnes','Rainfall_mm','Avg_Temperature_C',
    'Humidity_pct','Soil_Moisture_pct','Water_Used_m3',
    'Water_Efficiency_t_per_1000m3','Revenue_INR','Profit_INR','Disease_Pest_Risk_pct'
]
consolidated = df.groupby('Season')[key_indicators].mean().T
display(consolidated.round(2))

# Standardized seasonal profile to compare direction across variables
z = (consolidated - consolidated.mean(axis=1).values[:,None]) / consolidated.std(axis=1).replace(0,np.nan).values[:,None]
fig, ax = plt.subplots(figsize=(10,8))
sns.heatmap(z, annot=True, fmt='.2f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Standardized Seasonal Profile (Relative to the Three-Season Mean)')
ax.set_xlabel('Season'); ax.set_ylabel('Indicator')
plt.tight_layout(); plt.show()

# 14. Key Findings

The following cell generates concise findings directly from the computed results so that the final conclusions remain tied to the dataset.

In [ ]:
season_means = df.groupby('Season')[['Yield_Tonnes_Ha','Production_Tonnes','Revenue_INR','Profit_INR','Water_Used_m3','Water_Efficiency_t_per_1000m3','Disease_Pest_Risk_pct']].mean()

for metric in season_means.columns:
    best = season_means[metric].idxmax()
    worst = season_means[metric].idxmin()
    print(f'- {metric}: highest mean = {best} ({season_means.loc[best,metric]:,.2f}); lowest mean = {worst} ({season_means.loc[worst,metric]:,.2f})')

# strongest environmental/yield correlations
print('\nStrongest absolute environmental correlations with yield:')
display(env_yield.drop('Yield_Tonnes_Ha').sort_values(key=abs, ascending=False).head(5).to_frame('Correlation'))

print('\nStatistically significant seasonal differences at 5% level:')
display(test_results[(test_results['ANOVA_p']<0.05) | (test_results['Kruskal_p']<0.05)][['ANOVA_p','Kruskal_p']].round(6))

# 15. Conclusions

This project provides a focused seasonal analysis rather than a generic description of the entire dataset. The conclusions should be read together with the numerical tables and visualizations above.

- Agricultural performance differs across the three seasons in the supplied data, and the magnitude/direction of the difference depends on the indicator being considered.
- Environmental conditions such as rainfall, temperature, humidity and soil moisture vary by season, providing a measurable environmental context for agricultural outcomes.
- Resource usage and water efficiency also vary seasonally; therefore, production performance should not be judged using water consumption alone.
- Crop and state comparisons show that seasonal patterns may not be identical across all categories, so planning should account for local and crop-specific conditions.
- Economic outcomes can vary substantially, including negative-profit observations and extreme positive observations; both central tendency and variability are therefore important.
- Correlation analysis identifies relationships worth investigating, but these relationships should not be interpreted as proof of causation.
- Outlier analysis highlights unusual observations that deserve further investigation rather than automatic deletion.

# 16. Data-Driven Recommendations for Seasonal Agricultural Planning

Based on the analytical framework and observed dataset patterns, the following recommendations are appropriate:

1. **Use season-specific planning:** Avoid treating all seasons as operationally identical; use seasonal performance and environmental profiles when planning cultivation.
2. **Match crop choice to seasonal evidence:** Compare crop-by-season performance before selecting crops for a particular period.
3. **Monitor water efficiency:** Evaluate irrigation decisions using both water consumed and water-efficiency outcomes rather than water use alone.
4. **Use environmental indicators for preparedness:** Seasonal rainfall, temperature, humidity and soil-moisture patterns can be monitored as supporting indicators for agricultural planning.
5. **Review economic viability:** Consider expected cost, revenue and profit together because high production does not necessarily guarantee high profitability.
6. **Investigate unusual observations:** Extreme yield, production, profit or efficiency values should be validated and investigated before using them for planning benchmarks.
7. **Use region-specific strategies where needed:** If state-level seasonal rankings differ, planning should be adapted to local conditions instead of applying one nationwide strategy.
8. **Treat relationships as evidence for further study:** Correlations found in this observational dataset can guide future controlled or longitudinal studies, but should not be interpreted as causal effects.

# 17. Final Project Checklist Against the Internship Brief

| Internship requirement | Covered in notebook |
|---|---|
| Explore and understand dataset | Sections 5–6 |
| Clean and prepare data | Section 7 |
| Examine seasonal performance | Questions 1–2 |
| Identify seasonal patterns/trends | Questions 1–5 and consolidated comparison |
| Investigate environmental conditions vs outcomes | Questions 3 and 6 |
| Compare relevant groups | Question 5 and Question 9 |
| Examine resource usage | Question 4 and Question 12 |
| Examine economic outcomes | Question 8 |
| Identify significant differences | Question 10 |
| Identify unusual patterns | Question 11 |
| Apply statistical techniques | Question 10 |
| Apply visualization techniques | Throughout Questions 1–12 |
| Interpret findings based on evidence | Interpretation after each question |
| Develop conclusions | Section 15 |
| Provide data-driven recommendations | Section 16 |
| Document complete analysis in Jupyter Notebook | Entire notebook |

# 18. Final Note

This notebook is designed to be run from top to bottom in **Google Colab** using the supplied `seasonal_agriculture_performance_dataset.csv`. The analytical questions were selected after inspecting the actual dataset structure and are aligned with the internship brief's requirement to develop specific questions after exploration.

**End of Major Project – Seasonal Agriculture Performance Analysis**